In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.inference.predictor import RecyclingPredictor
from src.inference.camera import CameraInference
from src.utils.config import Config

In [2]:
import serial
import time

class ArduinoController:
    """
    Controla el Arduino Nano enviando instrucciones de clasificación (0-3)
    por comunicación serial USB.
    """
    def __init__(self, port='COM5', baudrate=9600, timeout=2):
        """
        Inicializar conexión con Arduino.
        
        Args:
            port: Puerto serial (ej. 'COM3' en Windows, '/dev/ttyUSB0' en Linux)
            baudrate: Velocidad de comunicación (9600 por defecto en Arduino)
            timeout: Timeout de lectura en segundos
        """
        try:
            self.ser = serial.Serial(port, baudrate, timeout=timeout)
            time.sleep(2)  # Esperar a que Arduino se reinicie tras conectar
            self.ser.reset_input_buffer()
            print(f"✓ Conectado a Arduino en puerto {port} a {baudrate} baud")
            
            # Leer mensaje inicial del Arduino
            if self.ser.in_waiting:
                msg = self.ser.readline().decode('utf-8', errors='ignore').strip()
                print(f"  Arduino: {msg}")
        except serial.SerialException as e:
            print(f"✗ Error conectando a Arduino: {e}")
            print("  Asegúrate de que el Arduino está conectado y el puerto es correcto")
            self.ser = None
    
    def send_instruction(self, class_id):
        """
        Enviar instrucción de clasificación al Arduino.
        
        Args:
            class_id: Identificador de clase (0-3)
                - 0: cardboard_paper
                - 1: ecoglasses
                - 2: metal_plastic
                - 3: trash
        
        Returns:
            bool: True si se envió correctamente, False si hubo error
        """
        if self.ser is None or not self.ser.is_open:
            print("✗ Arduino no conectado")
            return False
        
        if not isinstance(class_id, int) or class_id < 0 or class_id > 3:
            print(f"✗ ID de clase inválido: {class_id}. Debe estar entre 0 y 3")
            return False
        
        try:
            # Enviar como carácter ASCII ('0'-'3')
            self.ser.write(str(class_id).encode('utf-8'))
            print(f"→ Enviado: {class_id}")
            
            # Leer respuesta del Arduino
            response = self.ser.readline().decode('utf-8', errors='ignore').strip()
            if response:
                print(f"  Arduino: {response}")
            
            return True
        except Exception as e:
            print(f"✗ Error enviando instrucción: {e}")
            return False
    
    def close(self):
        """Cerrar conexión con Arduino."""
        if self.ser and self.ser.is_open:
            self.ser.close()
            print("✓ Conexión cerrada")

# Intentar conectar - CAMBIAR EL PUERTO SI ES NECESARIO
# En Windows: 'COM3', 'COM4', etc.
# En Linux: '/dev/ttyUSB0', '/dev/ttyACM0', etc.
# En Mac: '/dev/tty.usbserial-*'
try:
    arduino = ArduinoController(baudrate=9600)
except:
    print("Nota: Ajusta el puerto serial según tu sistema")
    arduino = None

✓ Conectado a Arduino en puerto COM5 a 9600 baud


In [3]:
config = Config('../configs/mobilenet_config.yaml')
model_path = '../outputs/checkpoints/best_model.pt'

class_mapping = {0: 'cardboard_paper', 1: 'ecoglasses', 2: 'metal_plastic', 3: 'trash'}

predictor = RecyclingPredictor(
    model_path=model_path,
    num_classes=config.model.get('num_classes'),
    architecture=config.model.get('architecture'),
    class_mapping=class_mapping
)
print("Predictor loaded.")

Predictor loaded.


C:\Users\tomas\OneDrive\Documents\GitHub\TP-final-vision\src\data\augmentation.py:67: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(


In [ ]:
from src.inference.camera_gradcam import CameraInferenceWithGradCAM

class CameraInferenceWithArduino(CameraInferenceWithGradCAM):
    """Extiende CameraInferenceWithGradCAM para enviar instrucciones al Arduino."""
    
    def __init__(self, predictor, arduino_controller, camera_id=0, enable_gradcam=True, 
                 stability_duration=4.0, stereo_mode=None, width=1280, height=1280,
                 black_threshold=0.7, brightness_threshold=40):
        super().__init__(predictor, camera_id=camera_id, enable_gradcam=enable_gradcam, 
                        stability_duration=stability_duration, stereo_mode=stereo_mode, 
                        width=width, height=height,
                        black_threshold=black_threshold, brightness_threshold=brightness_threshold)
        self.arduino = arduino_controller
    
    def process_frame(self, result):
        """Override de process_frame para incluir envío al Arduino cuando hay estabilidad."""
        if 'stable_class' in result:
            stable_class = result['stable_class']
            class_name = self.predictor.class_mapping.get(stable_class, f"Class {stable_class}")
            
            print(f"\n🎯 Clasificación ESTABLE: {class_name} (ID: {stable_class})")
            
            if self.arduino and self.arduino.ser:
                success = self.arduino.send_instruction(stable_class)
                if success:
                    self.last_sent_class = stable_class
                    print(f"✅ Enviado al Arduino")
            else:
                print(f"⚠️  Arduino no conectado")
                self.last_sent_class = stable_class

# 🔧 CONFIGURACIÓN
STEREO_MODE = 'left'  # 'left', 'right' para estéreo, None para mono
BLACK_THRESHOLD = 0.7  # 70% de pixeles negros = bandeja vacía
BRIGHTNESS_THRESHOLD = 40  # Brillo máximo para considerar pixel "negro"

if arduino and arduino.ser:
    print(f"🚀 Iniciando cámara con Arduino")
    print(f"📷 Modo: {STEREO_MODE or 'MONO'}")
    print(f"🎯 Detección vacío: {BLACK_THRESHOLD*100:.0f}% negro, brillo < {BRIGHTNESS_THRESHOLD}")
    
    camera_arduino = CameraInferenceWithArduino(
        predictor, arduino, camera_id=0, enable_gradcam=True,
        stability_duration=4.0, stereo_mode=STEREO_MODE, 
        width=2000, height=2000,
        black_threshold=BLACK_THRESHOLD,
        brightness_threshold=BRIGHTNESS_THRESHOLD
    )
    camera_arduino.run()
else:
    print("⚠ Arduino no disponible")
    camera = CameraInferenceWithGradCAM(
        predictor, camera_id=0, enable_gradcam=True, 
        stability_duration=4.0, stereo_mode=STEREO_MODE,
        black_threshold=BLACK_THRESHOLD,
        brightness_threshold=BRIGHTNESS_THRESHOLD
    )
    camera.run()

🚀 Iniciando cámara con Arduino - Modo: left
🚀 Cámara con Arduino - Modo: left


**Parámetros ajustables:**
- `black_threshold`: Porcentaje de pixeles oscuros para considerar vacío (default: 0.7 = 70%)
- `brightness_threshold`: Brillo máximo para considerar un pixel "negro" (default: 40 en escala 0-255)

In [ ]:
# Ejemplossss por si no anda bien: Probar diferentes configuraciones de detección de vacío

# Configuración MÁS SENSIBLE (detecta vacío más fácilmente)
# Usar si el sistema está clasificando el fondo negro como basura
# BLACK_THRESHOLD = 0.6  # 60% de negro
# BRIGHTNESS_THRESHOLD = 50  # Más tolerante con brillo

# Configuración MENOS SENSIBLE (más difícil detectar vacío)
# Usar si marca vacío cuando hay objeto
# BLACK_THRESHOLD = 0.8  # 80% de negro
# BRIGHTNESS_THRESHOLD = 30  # Más estricto con brillo

# Configuración ESTÁNDAR (recomendada para empezar)
BLACK_THRESHOLD = 0.7  # 70% de negro
BRIGHTNESS_THRESHOLD = 40  # Balance medio

In [6]:
arduino.close()

✓ Conexión cerrada
